In [13]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --- Path resolver: ищем папку analysis/notebooks ---
def find_notebooks_dir() -> Path:
    cwd = Path.cwd().resolve()

    # 1) Если мы уже в analysis/notebooks или глубже
    for p in [cwd, *cwd.parents]:
        if (p / "dataset").exists() and (p / "figures").exists():
            # это похоже на analysis/notebooks
            return p

    # 2) Ищем именно analysis/notebooks вверх по дереву
    for p in [cwd, *cwd.parents]:
        candidate = p / "analysis" / "notebooks"
        if candidate.exists():
            return candidate

    # 3) fallback: текущая директория
    return cwd

NOTEBOOKS_DIR = find_notebooks_dir()
DATASET_PATH = NOTEBOOKS_DIR / "dataset" / "yandex_music_data.json"
FIG_DIR = NOTEBOOKS_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("NOTEBOOKS_DIR:", NOTEBOOKS_DIR)
print("DATASET_PATH:", DATASET_PATH)
print("FIG_DIR:", FIG_DIR)

if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATASET_PATH}")

# --- Load data ---
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

tracks = pd.DataFrame(data.get("tracks", []))
likes = pd.DataFrame(data.get("likes", []))

likes["liked_at"] = pd.to_datetime(likes["liked_at"], utc=True, errors="coerce")

tracks["id"] = tracks["id"].astype("int64")
likes["track_id"] = likes["track_id"].astype("int64")

df = likes.merge(tracks, left_on="track_id", right_on="id", how="left")

# primary_genre fallback
df["primary_genre"] = df["primary_genre"].fillna(
    df["genres"].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None)
)

# main artist = first artist in artists[]
def first_artist(artists):
    if isinstance(artists, list) and len(artists) > 0:
        return artists[0].get("name")
    return None

df["main_artist"] = df["artists"].apply(first_artist)

def savefig(name: str):
    out = FIG_DIR / name
    plt.tight_layout()
    plt.savefig(out, dpi=180, bbox_inches="tight")
    plt.close()
    return out

df.head()

NOTEBOOKS_DIR: C:\Users\kesha\OneDrive\Рабочий стол\BigData\yandex-music-preference-analysis\analysis\notebooks
DATASET_PATH: C:\Users\kesha\OneDrive\Рабочий стол\BigData\yandex-music-preference-analysis\analysis\notebooks\dataset\yandex_music_data.json
FIG_DIR: C:\Users\kesha\OneDrive\Рабочий стол\BigData\yandex-music-preference-analysis\analysis\notebooks\figures


,track_id,liked_at,id,title,duration_ms,explicit,primary_genre,release_year,genres,artists,albums,main_artist
0,143449115,2025-10-03 15:02:01+00:00,143449115,MARTINE ROSE,186890,True,rusrap,2025.0,[rusrap],"[{'id': 13992820, 'name': 'madk1d'}, {'id': 82...","[{'id': 38435712, 'title': 'MARTINE ROSE', 'ge...",madk1d
1,330817,2025-10-03 09:52:49+00:00,330817,Let Down,299260,False,indie,1997.0,[indie],"[{'id': 36825, 'name': 'Radiohead'}]","[{'id': 3389007, 'title': 'OK Computer', 'genr...",Radiohead
2,10169820,2025-09-29 21:03:59+00:00,10169820,Alive,204540,False,electronics,2013.0,[electronics],"[{'id': 111191, 'name': 'Empire Of The Sun'}]","[{'id': 1182273, 'title': 'Ice On The Dune', '...",Empire Of The Sun
3,332895,2025-09-29 14:47:22+00:00,332895,We Are The People,267360,False,electronics,2008.0,[electronics],"[{'id': 111191, 'name': 'Empire Of The Sun'}]","[{'id': 51841, 'title': 'Walking On A Dream', ...",Empire Of The Sun
4,332764,2025-09-28 13:22:49+00:00,332764,Walking On A Dream,196340,False,electronics,2008.0,[electronics],"[{'id': 111191, 'name': 'Empire Of The Sun'}]","[{'id': 51841, 'title': 'Walking On A Dream', ...",Empire Of The Sun


In [14]:
# Топ-15 исполнителей
top_artists = df["main_artist"].value_counts().head(15).sort_values()

plt.figure(figsize=(10,6))
plt.barh(top_artists.index, top_artists.values)
plt.title("Топ-15 исполнителей по лайкам")
plt.xlabel("Количество лайков")
savefig("top_artists.png")


WindowsPath('C:/Users/kesha/OneDrive/Рабочий стол/BigData/yandex-music-preference-analysis/analysis/notebooks/figures/top_artists.png')

In [15]:
# Топ-15 жанров
top_genres = df["primary_genre"].value_counts().head(15).sort_values()

plt.figure(figsize=(10,6))
plt.barh(top_genres.index, top_genres.values)
plt.title("Топ-15 жанров (primary_genre) по лайкам")
plt.xlabel("Количество лайков")
savefig("top_genres.png")

WindowsPath('C:/Users/kesha/OneDrive/Рабочий стол/BigData/yandex-music-preference-analysis/analysis/notebooks/figures/top_genres.png')

In [16]:
# Динамика лайков по месяцам
monthly = df.set_index("liked_at").resample("MS").size()

plt.figure(figsize=(10,4))
plt.plot(monthly.index, monthly.values)
plt.title("Динамика лайков по месяцам")
plt.xlabel("Месяц")
plt.ylabel("Лайков")
savefig("likes_monthly.png")

WindowsPath('C:/Users/kesha/OneDrive/Рабочий стол/BigData/yandex-music-preference-analysis/analysis/notebooks/figures/likes_monthly.png')

In [17]:
# Тепловая карта активности (день недели × час)
tmp = df.dropna(subset=["liked_at"]).copy()
tmp["dow"] = tmp["liked_at"].dt.dayofweek  # 0=Mon
tmp["hour"] = tmp["liked_at"].dt.hour

pivot = tmp.pivot_table(index="dow", columns="hour", values="track_id", aggfunc="count", fill_value=0)

plt.figure(figsize=(12,4))
plt.imshow(pivot.values, aspect="auto")
plt.title("Тепловая карта активности (день недели × час) по лайкам")
plt.yticks(range(7), ["Пн","Вт","Ср","Чт","Пт","Сб","Вс"])
plt.xticks(range(0,24,2), [str(h) for h in range(0,24,2)])
plt.xlabel("Час")
plt.ylabel("День недели")
plt.colorbar(label="Лайков")
savefig("likes_heatmap_dow_hour.png")

WindowsPath('C:/Users/kesha/OneDrive/Рабочий стол/BigData/yandex-music-preference-analysis/analysis/notebooks/figures/likes_heatmap_dow_hour.png')

In [18]:
# Фильтрация «за период» + сохранение топов
def top_for_period(start: str, end: str, top_n: int = 10):
    start_dt = pd.to_datetime(start, utc=True)
    end_dt = pd.to_datetime(end, utc=True)

    d = df[(df["liked_at"] >= start_dt) & (df["liked_at"] <= end_dt)].copy()
    if d.empty:
        return None, None, d

    top_art = d["main_artist"].value_counts().head(top_n)
    top_gen = d["primary_genre"].value_counts().head(top_n)
    return top_art, top_gen, d

start = "2024-01-01"
end   = "2025-12-31"

top_art, top_gen, d = top_for_period(start, end, top_n=10)
print("Период:", start, "—", end)
print("Лайков в периоде:", len(d))

if top_art is not None:
    # Сохраним топы в CSV (удобно прикладывать к отчёту)
    out_dir = Path("../exports")
    out_dir.mkdir(parents=True, exist_ok=True)
    top_art.to_csv(out_dir / f"top_artists_{start}_{end}.csv", header=["likes"])
    top_gen.to_csv(out_dir / f"top_genres_{start}_{end}.csv", header=["likes"])

    # Графики топов за период
    plt.figure(figsize=(10,4))
    plt.bar(top_art.index[::-1], top_art.values[::-1])
    plt.xticks(rotation=45, ha="right")
    plt.title(f"Топ исполнителей за период {start} — {end}")
    plt.ylabel("Лайков")
    savefig(f"top_artists_{start}_{end}.png")

    plt.figure(figsize=(10,4))
    plt.bar(top_gen.index[::-1], top_gen.values[::-1])
    plt.xticks(rotation=45, ha="right")
    plt.title(f"Топ жанров за период {start} — {end}")
    plt.ylabel("Лайков")
    savefig(f"top_genres_{start}_{end}.png")


Период: 2024-01-01 — 2025-12-31
Лайков в периоде: 354
